# 5G Telecom AI Engine — Milestone 1: Environment Setup

This notebook verifies your Colab environment and prepares everything needed for subsequent milestones.

**Runtime:** GPU (T4 recommended) — Runtime > Change runtime type > T4 GPU

## Step 1 — Check GPU

In [ ]:
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — switch runtime to GPU!')

## Step 2 — Mount Google Drive

We use Drive to persist model checkpoints between Colab sessions (sessions reset after ~12h).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
GDRIVE_ROOT = '/content/drive/MyDrive/5G-LLM-ENGINE'
os.makedirs(f'{GDRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{GDRIVE_ROOT}/datasets', exist_ok=True)
print(f'Google Drive mounted. Project root: {GDRIVE_ROOT}')

## Step 3 — Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/cem8kaya/5G-LLM-ENGINE.git'
REPO_DIR = '/content/5G-LLM-ENGINE'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
os.system('ls -la')

## Step 4 — Install Dependencies

Installing all packages needed for training, RAG, and inference.

In [ ]:
# Install core packages — bitsandbytes must be installed before transformers
# to ensure 4-bit quantization (QLoRA) is available
!pip install -q bitsandbytes>=0.43.0
!pip install -q transformers>=4.40.0 datasets>=2.18.0
!pip install -q peft>=0.10.0 trl>=0.8.6 accelerate>=0.28.0
!pip install -q faiss-cpu sentence-transformers langchain langchain-community
!pip install -q fastapi uvicorn pydantic
!pip install -q rich click jsonschema rouge-score evaluate
print('All packages installed successfully.')

## Step 5 — Verify Environment

In [ ]:
import torch
import transformers
import peft
import trl
import bitsandbytes as bnb

print('=== Environment Verification ===')
print(f'Python:          {__import__("sys").version.split()[0]}')
print(f'PyTorch:         {torch.__version__}')
print(f'Transformers:    {transformers.__version__}')
print(f'PEFT:            {peft.__version__}')
print(f'TRL:             {trl.__version__}')
print(f'BitsAndBytes:    {bnb.__version__}')
print()
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM:            {total_mem:.1f} GB')
    print(f'BF16 supported:  {torch.cuda.is_bf16_supported()}')

## Step 6 — Project Structure Verification

In [ ]:
from pathlib import Path

EXPECTED_DIRS = [
    'dataset/raw', 'dataset/processed', 'dataset/schemas',
    'training/configs', 'training/checkpoints',
    'inference', 'rag/documents', 'rag/vectorstore',
    'utils', 'evaluation', 'api', 'notebooks', 'docs'
]

print('=== Project Structure Check ===')
all_ok = True
for d in EXPECTED_DIRS:
    p = Path(d)
    status = 'OK' if p.exists() else 'MISSING'
    if status == 'MISSING':
        all_ok = False
    print(f'  [{status}]  {d}')

print()
print('Milestone 1 PASSED' if all_ok else 'WARNING: Some directories missing — re-run setup.')

## Step 7 — Symlink Checkpoints to Google Drive

Colab VMs are ephemeral. We symlink the checkpoints directory to Drive so model saves persist across sessions.

In [ ]:
import os

LOCAL_CKPT = '/content/5G-LLM-ENGINE/training/checkpoints'
DRIVE_CKPT = '/content/drive/MyDrive/5G-LLM-ENGINE/checkpoints'

# Remove existing local dir (empty placeholder) and symlink to Drive
if os.path.isdir(LOCAL_CKPT) and not os.path.islink(LOCAL_CKPT):
    os.rmdir(LOCAL_CKPT)  # Only works if empty

if not os.path.exists(LOCAL_CKPT):
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f'Symlinked: {LOCAL_CKPT} -> {DRIVE_CKPT}')
else:
    print(f'Checkpoints already linked or exists: {LOCAL_CKPT}')

---
## Milestone 1 Complete

Your environment is ready. Next steps:

1. **Milestone 2** — Design telecom dataset schema and create 10+ realistic samples
2. Commit any local changes back to GitHub

```bash
cd /content/5G-LLM-ENGINE
git add -A && git commit -m 'milestone1: environment verified'
git push
```